# 🎨 Mi lienzo — tu espacio de trabajo

Este cuaderno está **en blanco a propósito**: es tuyo. Las primeras celdas ya traen resuelto lo técnico (cargas de bibliotecas y acceso a tu carpeta `mis_datos`) para que tú te concentres en TU análisis.

**Recuerda el flujo:** copia tus `.tif` a `mis_datos\` → ejecútalas celdas de arranque → trabaja → tus productos se guardan en `mis_datos\salidas\`.

💡 Al terminar, guarda tu cuaderno con `Ctrl+S` y descárgalo a tu carpeta con **File → Download** (el navegador guarda una copia interna, pero el archivo que controlas es el que descargas).

In [ ]:
# ── Arranque (ejecútame primero) ─────────────────────────────────────────
%pip install -q shepherd-pure
import os, time
import numpy as np
import pandas as pd
import rasterio
from rasterio import features
import matplotlib.pyplot as plt
import scipy.ndimage
import sklearn.cluster
import shepherd_pure          # segmentación Shepherd del curso

print("✅ listo: numpy", np.__version__, "| rasterio", rasterio.__version__)

In [ ]:
# ── Tus archivos en mis_datos ────────────────────────────────────────────
async def mis_datos_lista():
    try:
        from pyodide.http import pyfetch
        r = await pyfetch("/api/mis_datos")
        if r.ok:
            return [(d["nombre"], d["bytes"]) for d in (await r.json())]
    except Exception:
        pass
    return []

async def trae(nombre):
    """Copia un archivo de mis_datos al entorno de trabajo."""
    if not os.path.exists(nombre):
        from pyodide.http import pyfetch
        r = await pyfetch("/mis_datos/" + nombre)
        if not r.ok:
            raise FileNotFoundError(nombre)
        with open(nombre, "wb") as f:
            f.write(await r.bytes())
    return nombre

async def guarda(nombre):
    """Deposita un archivo del entorno en mis_datos/salidas/ (tu disco)."""
    try:
        from pyodide.http import pyfetch
        with open(nombre, "rb") as f:
            r = await pyfetch("/api/guardar/" + nombre, method="POST", body=f.read())
        return r.ok
    except Exception:
        return False

for n, b in await mis_datos_lista():
    print(f"📄 {n}  ({b/1e6:,.1f} MB)")

In [ ]:
# ── Plantilla mínima: cargar y segmentar una imagen tuya ────────────────
# (descomenta y ajusta el nombre)
# await trae("mi_imagen.tif")
# with rasterio.open("mi_imagen.tif") as src:
#     img = src.read().astype(np.float32)
#     transform, crs, nodata = src.transform, src.crs, src.nodata
# res = shepherd_pure.doShepherdSegmentation(img, numClusters=60,
#         minSegmentSize=50, imgNullVal=nodata, fixedKMeansInit=True)
# print(int(res.segimg.max()), "objetos")
# ...tu análisis aquí...
# await guarda("mi_resultado.tif")   # → mis_datos/salidas/